In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# ── Configuration ─────────────────────────────────────────────────────────────
BRONZE_BASE_PATH = "abfss://bronze@hospitaldatalake1.dfs.core.windows.net/Hospital_data/"
SILVER_BASE_PATH = "abfss://silver@hospitaldatalake1.dfs.core.windows.net/Hospital_data/"

TABLES = ["appointments", "billing", "doctors", "patients", "treatments"]

# ── Audit columns to drop from bronze (re-added fresh for silver) ───────────
AUDIT_COLS = ["_source_file", "_ingestion_ts", "_batch_date", "_pipeline_layer"]

# =============================================================================
# Per-table transformation functions
# =============================================================================

def transform_appointments(df):
    return (
        df
        .withColumn("appointment_date", F.to_date("appointment_date", "yyyy-MM-dd"))
        .withColumn("appointment_time", F.to_timestamp(F.col("appointment_time")))
        .withColumn("status",           F.initcap(F.trim(F.col("status"))))
        .withColumn("reason_for_visit", F.initcap(F.trim(F.col("reason_for_visit"))))
        .filter(F.col("appointment_id").isNotNull())
        .filter(F.col("patient_id").isNotNull())
        .filter(F.col("doctor_id").isNotNull())
        .dropDuplicates(["appointment_id"])
    )

def transform_billing(df):
    return (
        df
        .withColumn("bill_date",       F.to_date("bill_date", "yyyy-MM-dd"))
        .withColumn("amount",          F.col("amount").cast(DoubleType()))
        .withColumn("payment_method",  F.initcap(F.trim(F.col("payment_method"))))
        .withColumn("payment_status",  F.initcap(F.trim(F.col("payment_status"))))
        .filter(F.col("bill_id").isNotNull())
        .filter(F.col("amount") > 0)
        .dropDuplicates(["bill_id"])
    )

def transform_doctors(df):
    return (
        df
        .withColumn("years_experience", F.col("years_experience").cast(IntegerType()))
        .withColumn("first_name",       F.initcap(F.trim(F.col("first_name"))))
        .withColumn("last_name",        F.initcap(F.trim(F.col("last_name"))))
        .withColumn("specialization",   F.initcap(F.trim(F.col("specialization"))))
        .withColumn("hospital_branch",  F.initcap(F.trim(F.col("hospital_branch"))))
        .withColumn("email",            F.lower(F.trim(F.col("email"))))
        .withColumn("full_name",        F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
        .filter(F.col("doctor_id").isNotNull())
        .dropDuplicates(["doctor_id"])
    )

def transform_patients(df):
    return (
        df
        .withColumn("date_of_birth",     F.to_date("date_of_birth", "yyyy-MM-dd"))
        .withColumn("registration_date", F.to_date("registration_date", "yyyy-MM-dd"))
        .withColumn("first_name",        F.initcap(F.trim(F.col("first_name"))))
        .withColumn("last_name",         F.initcap(F.trim(F.col("last_name"))))
        .withColumn("gender",            F.upper(F.trim(F.col("gender"))))
        .withColumn("email",             F.lower(F.trim(F.col("email"))))
        .withColumn("insurance_provider", F.initcap(F.trim(F.col("insurance_provider"))))
        .withColumn("age",               F.floor(
            F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25
        ).cast(IntegerType()))
        .withColumn("full_name",         F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
        .filter(F.col("patient_id").isNotNull())
        .dropDuplicates(["patient_id"])
    )

def transform_treatments(df):
    return (
        df
        .withColumn("treatment_date", F.to_date("treatment_date", "yyyy-MM-dd"))
        .withColumn("cost",           F.col("cost").cast(DoubleType()))
        .withColumn("treatment_type", F.initcap(F.trim(F.col("treatment_type"))))
        .withColumn("description",    F.initcap(F.trim(F.col("description"))))
        .filter(F.col("treatment_id").isNotNull())
        .filter(F.col("cost") > 0)
        .dropDuplicates(["treatment_id"])
    )

TRANSFORM_MAP = {
    "appointments": transform_appointments,
    "billing":      transform_billing,
    "doctors":      transform_doctors,
    "patients":     transform_patients,
    "treatments":   transform_treatments,
}

# =============================================================================
# Process ALL tables: Bronze → Silver
# =============================================================================
for table_name in TABLES:
    print(f"\n{'='*60}")
    print(f"[SILVER] Processing: {table_name}")
    print(f"{'='*60}")

    # Step 1 : Read from Bronze (ADLS path, not metastore)
    bronze_df = spark.read.format("delta").load(f"{BRONZE_BASE_PATH}{table_name}")
    bronze_df = bronze_df.drop(*AUDIT_COLS)

    # Step 2 : Apply table-specific transformations
    silver_df = TRANSFORM_MAP[table_name](bronze_df)

    # Step 3 : Add silver audit columns
    silver_df = (
        silver_df
        .withColumn("_silver_ts",      F.current_timestamp())
        .withColumn("_pipeline_layer", F.lit("SILVER"))
        .withColumn("_batch_date",     F.current_date())
    )

    row_count = silver_df.count()
    print(f"[SILVER] Row count: {row_count}")

    # Step 4 : Write to Silver Delta
    silver_path = f"{SILVER_BASE_PATH}{table_name}"
    silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)

    print(f"[SILVER] Written to: {silver_path}")

print(f"\n{'='*60}")
print("[SILVER] ALL tables processed successfully!")
print(f"{'='*60}")


[SILVER] Processing: appointments
[SILVER] Row count: 200
[SILVER] ✅ Written to: abfss://silver@hospitaldatalake1.dfs.core.windows.net/Hospital_data/appointments

[SILVER] Processing: billing
[SILVER] Row count: 200
[SILVER] ✅ Written to: abfss://silver@hospitaldatalake1.dfs.core.windows.net/Hospital_data/billing

[SILVER] Processing: doctors
[SILVER] Row count: 10
[SILVER] ✅ Written to: abfss://silver@hospitaldatalake1.dfs.core.windows.net/Hospital_data/doctors

[SILVER] Processing: patients
[SILVER] Row count: 50
[SILVER] ✅ Written to: abfss://silver@hospitaldatalake1.dfs.core.windows.net/Hospital_data/patients

[SILVER] Processing: treatments
[SILVER] Row count: 200
[SILVER] ✅ Written to: abfss://silver@hospitaldatalake1.dfs.core.windows.net/Hospital_data/treatments

[SILVER] ✅ ALL tables processed successfully!
